# Exercise 03, Part 2: What is wrong with these files?

Run this in Google Colab, signed in with your WashU Google account.

Every file here is real course data. Two came off a survey crew's controller,
one off a drone, and the rest were written out of those by ordinary software.
Each is missing something or states something false, and none of them will
raise an error when you open it.

For each file, work out what it actually holds, say what the discrepancy costs
in meters, and write down what the file should have recorded in the first place.
The last of those is what you hand in.

There is no map to make here. The answers are numbers and sentences.

## Setup

This installs the geospatial stack and downloads the data bundle. Let it finish
before you go on.

In [ ]:
!pip -q install pyproj rasterio geopandas
print("installed")

In [ ]:
import pathlib, urllib.request, urllib.error

REPO = "https://raw.githubusercontent.com/washu-eeps/eeps4684-ex03-data/main/data"
FILES = [
    "tisch_control.csv",
    "benchmark_coordinates.csv",
    "tisch_control_4326.gpkg",
    "benchmarks.gpkg",
    "benchmarks.shp",
    "benchmarks.shx",
    "benchmarks.dbf",
    "benchmarks.prj",
    "benchmarks.cpg",
    "tisch_2024_band5_30cm.tif"
]

pathlib.Path("data").mkdir(exist_ok=True)
missing = []
for name in FILES:
    dest = pathlib.Path("data") / name
    if dest.exists():
        continue
    try:
        urllib.request.urlretrieve(f"{REPO}/{name}", dest)
    except urllib.error.HTTPError:
        missing.append(name)

if missing:
    print("Could not download:", ", ".join(missing))
    print("Download the Exercise 03 data bundle from Canvas, unzip it, and")
    print("upload the files with the next cell instead.")
else:
    for p in sorted(pathlib.Path("data").iterdir()):
        print(f"{p.stat().st_size/1e6:8.2f} MB  {p.name}")

In [ ]:
# ONLY run this if the download above failed. Select every file in the bundle
# at once; the picker accepts a multiple selection.
import pathlib, shutil
from google.colab import files

pathlib.Path("data").mkdir(exist_ok=True)
for name in files.upload():
    shutil.move(name, pathlib.Path("data") / name)
    print("stored data/" + name)

## Step 1. Which reference frame is the control file in?

`tisch_control.csv` is the control survey the class works from. Open it and see
what it tells you.

In [ ]:
import pandas as pd

control = pd.read_csv("data/tisch_control.csv")
print(control.head().to_string(index=False))
print()
print(f"{len(control)} points")
print(f"Easting  {control.Easting.min():.1f} to {control.Easting.max():.1f}")
print(f"Northing {control.Northing.min():.1f} to {control.Northing.max():.1f}")
print(f"site spans {control.Easting.max()-control.Easting.min():.0f} m east-west")

Five columns, and none of them is a reference frame. No units column, no epoch,
and nothing saying whether `Elevation` is measured from the ellipsoid or from
sea level.
[Metadata and provenance](https://bradleylab.github.io/geospatial-field-methods/docs/toolchain/metadata-and-provenance)
makes the point: a coordinate is three numbers, and a position is a location in
some frame, so this file does not yet hold positions.

The file still gives you enough to establish the frame. Start by ruling one out.
Both projected systems in use around here are in meters, so the numbers are
plausible in either.

In [ ]:
from pyproj import Transformer

# Two candidate frames, both NAD83-family, both in meters, both used in Missouri.
CANDIDATES = {
    "EPSG:6512":  "NAD83(2011) / Missouri East",
    "EPSG:26915": "NAD83 / UTM zone 15N",
}

for epsg, name in CANDIDATES.items():
    lon, lat = Transformer.from_crs(epsg, "EPSG:6318", always_xy=True).transform(
        control.Easting.to_numpy(), control.Northing.to_numpy())
    print(f"read as {name:28s} -> {lat.mean():8.4f} N, {lon.mean():9.4f} E")

That is check 1 from
[coordinate systems in practice](https://bradleylab.github.io/geospatial-field-methods/docs/foundations/crs-in-practice):
*is it on the right continent?* One candidate puts the survey on the Danforth
campus. The other puts it several hundred kilometers out in the Pacific, a
failure too large to overlook.

The second candidate failed loudly, but that only leaves the first one
plausible. Proving it needs an independent measurement of the same ground, and
you have one.

`benchmark_coordinates.csv` holds four monumented benchmarks: marks that were
surveyed and published, so unlike the control points there is an outside answer
for them.

In [ ]:
bench = pd.read_csv("data/benchmark_coordinates.csv")
print(list(bench.columns))
print()
print(bench[["OID_", "Easting", "Northing", "NAD83_2011_X", "NAD83_2011_Y",
             "EPSG6318_X", "EPSG6318_Y", "ElvGeo03A"]].to_string(index=False))

This file has the opposite problem. It carries the same four points in several
frames at once and still never says which column is which frame. The redundancy
is what makes it useful. If two of these benchmarks also appear in the control
file, the two files describe the same physical marks, and you can test a guess
about the control file's frame by seeing whether it closes.

The next cell looks for them.

In [ ]:
import numpy as np

# Convert the control points to NAD83(2011) latitude and longitude, ASSUMING
# the frame you did not rule out, then look for benchmarks nearby.
lon, lat = Transformer.from_crs("EPSG:6512", "EPSG:6318", always_xy=True).transform(
    control.Easting.to_numpy(), control.Northing.to_numpy())

for _, b in bench.iterrows():
    m_per_deg_e = 111_320 * np.cos(np.radians(b.EPSG6318_Y))
    d = np.hypot((lon - b.EPSG6318_X) * m_per_deg_e,
                 (lat - b.EPSG6318_Y) * 111_320)
    i = int(d.argmin())
    near = f"{d[i]*1000:6.1f} mm" if d[i] < 1 else f"{d[i]:6.1f} m "
    print(f"benchmark {int(b.OID_)}: nearest control point {control.Name[i]:5s} "
          f"at {near}   ({control.Description[i]})")

Two of the four benchmarks land within a centimeter of a named control point.
The other two sit more than a hundred meters from anything, because they are
outside the surveyed area. A one-centimeter closure between two files produced
by different people at different times is a survey-grade agreement, and it
establishes the control file's frame.

Report the two matches, the closure in millimeters, and the EPSG code you have
proved. Then say what would have happened if you had guessed the other candidate
and never run the check.

## Step 2. What are the units, and what does getting them wrong cost?

Look again at the benchmark file. Its `Easting`/`Northing` pair and its
`NAD83_2011_X`/`_Y` pair are the same four points in the same projection. They
do not look alike because they are not in the same unit.

You can measure which unit it is rather than taking it on trust.

In [ ]:
ratio_e = (bench.Easting / bench.NAD83_2011_X).to_numpy()
ratio_n = (bench.Northing / bench.NAD83_2011_Y).to_numpy()

print(f"easting  ratio {ratio_e.mean():.7f}   spread {ratio_e.std():.1e}")
print(f"northing ratio {ratio_n.mean():.7f}   spread {ratio_n.std():.1e}")
print()
print(f"US survey foot  1200/3937 m  ->  1 m = {3937/1200:.7f} ft")
print(f"international foot 0.3048 m ->  1 m = {1/0.3048:.7f} ft")

The two candidate feet differ in the sixth decimal place, which is two parts per
million. That is a small fraction of a large number: a State Plane easting runs
to six figures, so the absolute error is big enough to trip over.

Read the file's feet as the *wrong* foot and see where the point moves.

In [ ]:
INTL_FT = 0.3048              # exact, by definition
US_FT = 1200 / 3937           # exact, by definition

correct = bench.Easting * US_FT
wrong = bench.Easting * INTL_FT

for _, row in pd.DataFrame({"oid": bench.OID_, "ft": bench.Easting,
                            "correct_m": correct, "wrong_m": wrong}).iterrows():
    print(f"benchmark {int(row.oid)}: {row.ft:12.3f} ft -> "
          f"{row.correct_m:11.3f} m correct, {row.wrong_m:11.3f} m wrong, "
          f"off by {row.correct_m - row.wrong_m:+.3f} m")

Answer in your report: how large is the error, and how does it compare to the
RTK precision you measured in Exercise 01?

A datum problem and a units problem both show up as a systematic offset, and
over a site 250 m across both look like a constant shift. Given only a set of
shifted coordinates, what test tells them apart? The clue is in how the offset
changes between the four benchmarks.

## Step 3. How far apart are NAD83 and WGS84 at this site?

In Exercise 01 you converted a position to WGS84 and reported the accuracy the
tool claimed for the transformation. What you took on trust was the size of the
shift itself. Here you measure it, because the benchmark file carries the same
four marks in both frames.

`EPSG6318_X`/`_Y` are NAD83(2011). `WGS84_LON_dd`/`_LAT_dd` are the same four
marks in WGS84. Subtract them.

In [ ]:
lat0 = bench.EPSG6318_Y.to_numpy()
m_per_deg_e = 111_320 * np.cos(np.radians(lat0))

de = (bench.WGS84_LON_dd.to_numpy() - bench.EPSG6318_X.to_numpy()) * m_per_deg_e
dn = (bench.WGS84_LAT_dd.to_numpy() - bench.EPSG6318_Y.to_numpy()) * 111_320

print("WGS84 minus NAD83(2011), from the file's own columns:")
for oid, e, n in zip(bench.OID_, de, dn):
    print(f"  benchmark {int(oid)}: east {e:+.3f} m  north {n:+.3f} m  "
          f"total {np.hypot(e, n):.3f} m")

Just under a meter, pointing the same way at all four marks. That is the datum
shift, measured from your own data.

Check it by an independent route, which is check 4 from the reference text: ask
PROJ for the same transformation and see whether it agrees.

In [ ]:
# WGS84 is a family of realizations, not one frame. Ask for a specific one.
to_g1762 = Transformer.from_crs("EPSG:6318", "EPSG:9057", always_xy=True)
lo2, la2 = to_g1762.transform(bench.EPSG6318_X.to_numpy(), lat0)

pe = (lo2 - bench.EPSG6318_X.to_numpy()) * m_per_deg_e
pn = (la2 - lat0) * 111_320

print(f"PROJ, NAD83(2011) -> WGS84 (G1762)")
print(f"   applies      east {pe.mean():+.3f} m  north {pn.mean():+.3f} m  "
      f"total {np.hypot(pe, pn).mean():.3f} m")
print(f"   claims accuracy {to_g1762.accuracy} m")
print(f"   disagrees with the file by {np.hypot(de-pe, dn-pn).mean()*1000:.1f} mm")

The surveyor's software and PROJ agree to the millimeter on a shift of nearly a
meter. Two independent routes to the same answer is a strong check, and it is
not run often.

Next, ask for plain "WGS84" without naming a realization, which is what most
workflows do.

In [ ]:
for dst, label in [("EPSG:9057", 'WGS84 (G1762), a named realization'),
                   ("EPSG:4326", 'WGS 84, the generic code')]:
    t = Transformer.from_crs("EPSG:6318", dst, always_xy=True)
    lo2, la2 = t.transform(bench.EPSG6318_X.to_numpy(), lat0)
    shift = np.hypot((lo2 - bench.EPSG6318_X.to_numpy()) * m_per_deg_e,
                     (la2 - lat0) * 111_320).mean()
    print(f"{label:36s} applies {shift:.3f} m, claims accuracy {t.accuracy} m")
    print(f"{'':36s} operation: {t.description[:80]}")
    print()

Read that output twice.

Ask for the named realization and PROJ moves the point most of a meter, and
reports the result as good to a centimeter. Ask for generic `EPSG:4326` and PROJ
moves the point not at all, and reports it as good to two meters. Same input,
same software, one word different in the request.

Neither answer is a bug. `EPSG:4326` does not identify a realization precisely
enough to compute a shift, so PROJ declines to apply one and widens the stated
uncertainty to cover the range. It declines *silently*, though: nothing fails,
and the number that comes back looks as authoritative as the good one.

Answer in your report: which of the two would you use, and what would you have
to state alongside it? And which of the two do you think most published "WGS84"
coordinates actually are?

## Step 4. A file that states its frame, incorrectly

`tisch_control_4326.gpkg` is the control survey again, converted to latitude and
longitude and saved as a GeoPackage. Unlike the CSVs, a GeoPackage has a proper
field for the reference frame, and this one is filled in.

It is filled in wrongly: the coordinates are NAD83(2011) and the file says
WGS 84.

In [ ]:
import geopandas as gpd

g = gpd.read_file("data/tisch_control_4326.gpkg")
print(f"declared CRS : {g.crs.to_string()}  ({g.crs.name})")
print(f"features     : {len(g)}")
print(g.head(3)[["Name", "geometry"]].to_string(index=False))

Nothing complains. Project it back to the frame you proved in Step 1, twice:
once believing the label, and once knowing the truth.

In [ ]:
lon_g, lat_g = g.geometry.x.to_numpy(), g.geometry.y.to_numpy()

believe = Transformer.from_crs("EPSG:4326", "EPSG:6512", always_xy=True)
truth = Transformer.from_crs("EPSG:6318", "EPSG:6512", always_xy=True)

be, bn = believe.transform(lon_g, lat_g)
te, tn = truth.transform(lon_g, lat_g)

print(f"believing the label : accuracy {believe.accuracy} m")
print(f"knowing the truth   : accuracy {truth.accuracy} m")
print(f"coordinates differ by {np.hypot(be-te, bn-tn).max()*1000:.3f} mm")
print()
print(f"and the correct route reproduces tisch_control.csv to "
      f"{np.hypot(te-control.Easting.to_numpy(), tn-control.Northing.to_numpy()).max()*1000:.2f} mm")

The obvious reading of that output is wrong, so take this one slowly.

The coordinates are identical to the micrometer, which makes it tempting to
conclude the mislabel cost nothing.

What changed is the stated accuracy of the transformation, and that figure is
part of your result. Write down both numbers and say in one sentence what the
mislabel destroyed, given that it was not the coordinate.

Then say what this implies for a dataset that has passed through several hands.
If the label is wrong and the numbers never move, at what point in the chain
could anyone have caught it?

## Step 5. Undeclared nodata

Now a raster. `tisch_2024_band5_30cm.tif` is one band of the five-band
orthophoto flown over campus by a previous class in October 2024, resampled to
30 cm so it fits in this notebook.

The flight covered an irregular polygon, but a raster is a rectangle, so the
corners outside the flight have to hold something.

In [ ]:
import rasterio

with rasterio.open("data/tisch_2024_band5_30cm.tif") as src:
    band = src.read(1)
    print(f"size          {src.width} x {src.height}")
    print(f"dtype         {src.dtypes[0]}")
    print(f"CRS           {src.crs}")
    print(f"declared nodata {src.nodata}")
    print(f"band name     {src.descriptions[0]}")

The file declares no nodata value, so every reader treats every cell as a
measurement. Find out what is in the corners and how much of the grid they
cover.

In [ ]:
values, counts = np.unique(band, return_counts=True)
fill = values[counts.argmax()]

print(f"most common value: {fill}, in {counts.max():,} cells "
      f"({100*counts.max()/band.size:.1f}% of the grid)")
print()
print(f"mean, fill counted as data : {band.mean():10.1f}")
print(f"mean, fill excluded        : {band[band != fill].mean():10.1f}")
print(f"the undeclared fill biases the mean "
      f"{100*(band[band != fill].mean()-band.mean())/band[band != fill].mean():.1f}% low")

Answer in your report: roughly a seventh of that grid is not data, and the
band's mean brightness is wrong by about the same fraction. Give the correct
mean, and the one line in the file's header that would have prevented the error.

This is band 5 of 5, and its name in the file is `Band_5`. Nothing in the file
records what wavelength it holds. In Session 9 you will compute vegetation
indices, which are ratios of specific bands, from a file like this one. What
would you have to find out first, where would you look, and what happens to the
index if you guess wrong?

## Step 6. What a shapefile drops

Two files, `benchmarks.gpkg` and `benchmarks.shp`, hold the same four benchmarks
written out of the same table by the same command. Compare their columns.

In [ ]:
gpkg = gpd.read_file("data/benchmarks.gpkg")
shp = gpd.read_file("data/benchmarks.shp")

print(f"{'GeoPackage':22s}  {'shapefile':22s}")
for a, b in zip(gpkg.columns, shp.columns):
    flag = "   <-- renamed" if a != b else ""
    print(f"{a:22s}  {b:22s}{flag}")
print()
print(f"are the values intact? "
      f"{np.allclose(gpkg.NAD83_2011_X, shp.NAD83_2011)}")

The shapefile format caps attribute names at ten characters. The writer did not
refuse the longer ones: it shortened them, and where shortening produced a
collision it appended a digit to keep them distinct.

Answer in your report: every value survived, so given only the shapefile, which
of the two renamed coordinate columns is the easting and which is the northing?
Say how you would decide, and whether your method would still work at a site
where easting and northing are similar numbers.

Then count how many files each format wrote. Look in `data/`.

## Step 7. Write the record that should have shipped with the data

This is the part you hand in.

Pick `tisch_control.csv` and write the metadata record it should have carried.
The
[minimum record table](https://bradleylab.github.io/geospatial-field-methods/docs/toolchain/metadata-and-provenance)
in the reference text lists what has to be answerable. Fill in every line you
can establish from the work above or from the Exercise 01 materials. For every
line you cannot, write that you cannot and say what you would have to ask. A
blank is a gap; "unknown, would have to ask the survey crew" is a record.

Be careful with the height column. You proved the horizontal frame, but you have
proved nothing about `Elevation`, and there is a specific reason the benchmark
file cannot settle it for you.

In [ ]:
# Write your record here, then run the cell to save it. This is a deliverable:
# download it and attach it to your report.
record = '''
tisch_control.csv
=================

What was measured :
Instrument        :
Configuration     :
When              :
Where             :
Horizontal datum  :
Projection        :
Units             :
Epoch             :
Vertical datum    :
Geoid model       :
Corrections       :
Claimed accuracy  :
Observer          :
Processing so far  :

Established how:

Still unknown, and who to ask:
'''

with open("tisch_control.csv.txt", "w") as f:
    f.write(record.strip() + "\n")
print(record)

In [ ]:
from google.colab import files
files.download("tisch_control.csv.txt")

## What you should have by the end

You should end with six numbers and one file.

The EPSG code of the control file and the closure in millimeters that proves it.
The units of the benchmark file and what the wrong foot costs in meters. The
NAD83-to-WGS84 shift at this site, measured twice. The two stated accuracies
from Step 4. The corrected mean of the raster band. And the metadata record.

Carry them into the report. Part 3 goes back to a GUI for the two jobs a GUI
does better, and the georeferencing residuals you get there need the distinction
from Step 4 to interpret.